In [17]:
# Housing Price Regression Model - Time Series
# =============================================
# 
# Predicts future average rent per zip code using:
# - Time series ZORI rent data (2015-2025)
# - Expanded gentrification features from Yelp

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.append(str(Path.cwd().parent))

from time_series_preparation import TimeSeriesDataPreparation
from regression_model import HousingPriceRegressor

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)


## Step 1: Data Preparation

Load and merge property data (ZORI rent) with gentrification data from Yelp.


In [18]:
# Initialize data preparation
prep = DataPreparation(
    data_dir="../../first-draft/raw_data",
    eda_dir="../../eda"
)

# Prepare and save data
processed_data = prep.save_processed_data("../data/processed_data.csv")

# Verify data structure
print("\n" + "="*60)
print("DATA VERIFICATION")
print("="*60)
print(f"\nDataset shape: {processed_data.shape}")
print(f"\nColumns in saved data: {list(processed_data.columns)}")
print(f"\n'zip' column present: {'zip' in processed_data.columns}")
print(f"'target' column present: {'target' in processed_data.columns}")

if 'zip' in processed_data.columns:
    print(f"\nUnique zip codes: {processed_data['zip'].nunique()}")
    print(f"Zip codes: {sorted(processed_data['zip'].unique())}")
else:
    print("\n⚠ WARNING: 'zip' column is missing from processed data!")

# Display summary
print("\n" + "="*60)
print("DATA SUMMARY")
print("="*60)
print(f"\nFeatures ({len(prep.feature_names)}):")
for i, col in enumerate(prep.feature_names, 1):
    print(f"  {i:2d}. {col}")

# Display first few rows
print("\nFirst few rows:")
processed_data.head()



Preparing data for modeling...

Merging data sources...
Loading ZORI rent data...
✓ Loaded ZORI data: 11 zip codes
Loading gentrification data...


KeyError: 'zip'

## Step 2: Exploratory Data Analysis

Explore the relationship between gentrification features and housing prices.


In [ ]:
# Load processed data for EDA
data = pd.read_csv("../data/processed_data.csv")

# Verify data structure
print("="*60)
print("DATA VERIFICATION")
print("="*60)
print(f"Columns in loaded data: {list(data.columns)}")
print(f"'zip' column present: {'zip' in data.columns}")
print(f"'target' column present: {'target' in data.columns}")

if 'zip' not in data.columns:
    raise ValueError("'zip' column is missing from processed data! Check data preparation.")

# Basic statistics
print("\n" + "="*60)
print("DESCRIPTIVE STATISTICS")
print("="*60)
print(f"\nTarget variable (latest_rent):")
print(data['target'].describe())

# Correlation analysis
numeric_cols = data.select_dtypes(include=[np.number]).columns
correlations = data[numeric_cols].corr()['target'].sort_values(ascending=False)

print("\n" + "="*60)
print("TOP CORRELATIONS WITH HOUSING PRICE")
print("="*60)
print(correlations.head(10))
print("\nBottom correlations:")
print(correlations.tail(10))


In [ ]:
# Visualize relationships
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Rent vs Gentrification Score
if 'gentrification_score' in data.columns:
    axes[0, 0].scatter(data['gentrification_score'], data['target'], alpha=0.7)
    axes[0, 0].set_xlabel('Gentrification Score')
    axes[0, 0].set_ylabel('Housing Price ($)')
    axes[0, 0].set_title('Housing Price vs Gentrification Score')
    axes[0, 0].grid(True, alpha=0.3)

# 2. Rent vs Gentrification Rate
if 'gentrification_rate' in data.columns:
    axes[0, 1].scatter(data['gentrification_rate'], data['target'], alpha=0.7)
    axes[0, 1].set_xlabel('Gentrification Rate')
    axes[0, 1].set_ylabel('Housing Price ($)')
    axes[0, 1].set_title('Housing Price vs Gentrification Rate')
    axes[0, 1].grid(True, alpha=0.3)

# 3. Rent vs Business Count
if 'total_business_count' in data.columns:
    axes[1, 0].scatter(data['total_business_count'], data['target'], alpha=0.7)
    axes[1, 0].set_xlabel('Total Business Count')
    axes[1, 0].set_ylabel('Housing Price ($)')
    axes[1, 0].set_title('Housing Price vs Business Count')
    axes[1, 0].grid(True, alpha=0.3)

# 4. Rent vs Average Rating
if 'avg_business_rating' in data.columns:
    axes[1, 1].scatter(data['avg_business_rating'], data['target'], alpha=0.7)
    axes[1, 1].set_xlabel('Average Business Rating')
    axes[1, 1].set_ylabel('Housing Price ($)')
    axes[1, 1].set_title('Housing Price vs Business Rating')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/eda_relationships.png', dpi=300, bbox_inches='tight')
plt.show()


## Step 3: Train Regression Models

Train multiple regression models and compare their performance.


In [ ]:
# Initialize regressor
regressor = HousingPriceRegressor(random_state=42)

# Run full modeling pipeline
results_df, importance_df = regressor.run_full_pipeline(
    data_path="../data/processed_data.csv",
    output_dir="../outputs"
)

# Display results
print("\n" + "="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)
print(results_df[['model', 'test_rmse', 'test_mae', 'test_r2', 'cv_rmse_mean']].to_string(index=False))


## Step 4: Feature Importance Analysis

Understand which features are most predictive of housing prices.


In [ ]:
# Display feature importance
print("="*60)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*60)
print("\nTop 15 Most Important Features:")
print(importance_df.head(15).to_string(index=False))

# Visualize feature importance
plt.figure(figsize=(12, 8))
top_15 = importance_df.head(15)
plt.barh(range(len(top_15)), top_15['importance'], color='steelblue')
plt.yticks(range(len(top_15)), top_15['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 15 Most Important Features for Housing Price Prediction')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/feature_importance_detailed.png', dpi=300, bbox_inches='tight')
plt.show()


## Step 5: Model Interpretation

Analyze predictions and residuals to understand model performance.


In [ ]:
# Load test data for detailed analysis
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

data = pd.read_csv("../data/processed_data.csv")
X = data.drop(columns=['target', 'zip'] + 
              [col for col in data.columns if col in 
               ['RegionID', 'RegionName', 'RegionType', 'StateName', 
                'State', 'City', 'Metro', 'CountyName', 'city', 'year']])
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = RobustScaler()
X_test_scaled = pd.DataFrame(
    scaler.fit_transform(X_train).transform(X_test),
    columns=X.columns,
    index=X_test.index
)

# Get predictions from best model
y_pred = regressor.best_model.predict(X_test_scaled)

# Create detailed analysis dataframe
analysis_df = pd.DataFrame({
    'zip': data.loc[X_test.index, 'zip'].values if 'zip' in data.columns else None,
    'actual': y_test.values,
    'predicted': y_pred,
    'residual': y_test.values - y_pred,
    'error_pct': ((y_test.values - y_pred) / y_test.values * 100)
})

print("="*60)
print("PREDICTION ANALYSIS")
print("="*60)
print(f"\nMean Absolute Error: ${analysis_df['residual'].abs().mean():.2f}")
print(f"Mean Absolute Percentage Error: {analysis_df['error_pct'].abs().mean():.2f}%")
print(f"\nPredictions Summary:")
print(analysis_df[['actual', 'predicted', 'residual', 'error_pct']].describe())

analysis_df.head(10)


## Step 6: Conclusions

Summarize findings and insights from the regression model.


In [ ]:
print("="*60)
print("KEY FINDINGS")
print("="*60)

print(f"\n1. Best Model: {regressor.best_model_name}")
best_result = results_df[results_df['model'] == regressor.best_model_name].iloc[0]
print(f"   - Test RMSE: ${best_result['test_rmse']:.2f}")
print(f"   - Test R²: {best_result['test_r2']:.4f}")
print(f"   - Test MAE: ${best_result['test_mae']:.2f}")

print(f"\n2. Top 3 Most Important Features:")
for i, row in importance_df.head(3).iterrows():
    print(f"   {i+1}. {row['feature']}: {row['importance']:.4f}")

print(f"\n3. Gentrification Features Impact:")
gent_features = importance_df[importance_df['feature'].str.contains('gentrification|business', case=False, na=False)]
if len(gent_features) > 0:
    print(f"   - {len(gent_features)} gentrification-related features in top predictors")
    print(f"   - Average importance: {gent_features['importance'].mean():.4f}")

print(f"\n4. Model Performance:")
print(f"   - The model explains {best_result['test_r2']*100:.1f}% of variance in housing prices")
print(f"   - Average prediction error: ${best_result['test_mae']:.2f}")

print("\n" + "="*60)
print("CONCLUSIONS")
print("="*60)
print("""
1. Gentrification features from Yelp data are predictive of housing prices
2. Business density and quality metrics correlate with higher housing prices
3. The regression model successfully combines property and gentrification data
4. Geographic and temporal features also play important roles
""")
